# Monolithic d-Ape — Full Pipeline
Self-supervised stereo depth estimation, bin-based head, ConvNeXt V2 backbone, HOG late-fusion, GNN sparse-anchor refinement.

In [ ]:
!pip install torch-geometric --break-system-packages -q

In [1]:
# ============================================================
# CELL 1: CONFIG
# ============================================================
import os
import math
import random
import glob
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import cv2
from PIL import Image
import torchvision.transforms as T

CFG = {
    "seed": 42,
    "img_h": 320,
    "img_w": 1024,
    "batch_size": 6,
    "num_workers": 2,
    "epochs": 20,
    "lr_head": 1e-4,
    "lr_backbone": 1e-5,
    "freeze_backbone_epochs": 3,
    "backbone_name": "convnextv2_nano",
    "n_bins": 64,
    "min_depth": 0.1,
    "max_depth": 100.0,
    "smoothness_weight": 1e-3,
    "lr_consistency_weight": 1.0,
    "ssim_weight": 0.85,
    "bin_entropy_weight": 0.05,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "amp": True,
    "data_root": "/kaggle/input/datasets/klemenko/kitti-dataset",
    "work_dir": "/kaggle/working",
    "ckpt_dir": "/kaggle/working/ckpts",
    "sift_cache_dir": "/kaggle/working/sift_cache",
    "val_ratio": 0.1,
    "test_ratio": 0.1,
    "log_every": 50,
    "save_every": 5,
    "use_hog": True,
    "use_gnn_refine": True,
    "gnn_grid_stride": 8,
    "gnn_k_neighbors": 8,
}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])
os.makedirs(CFG["ckpt_dir"], exist_ok=True)
os.makedirs(CFG["sift_cache_dir"], exist_ok=True)
print("device:", CFG["device"])

device: cuda


In [ ]:
# ============================================================
# CELL 2: CALIB PARSER + BUILD TRAIN/VAL/TEST LIST
# ============================================================
def parse_calib(calib_path):
    with open(calib_path, "r") as f:
        lines = f.readlines()
    calib = {}
    for line in lines:
        if ":" not in line:
            continue
        key, val = line.split(":", 1)
        key = key.strip()
        if key in ("P2", "P3"):
            calib[key] = np.array([float(x) for x in val.strip().split()])
    P2, P3 = calib["P2"], calib["P3"]
    fx = P2[0]
    baseline = abs(P2[3] - P3[3]) / fx
    return fx, baseline


def build_list_file(data_root, out_path, split="training"):
    calib_dir = os.path.join(data_root, "data_object_calib", split, "calib")
    left_dir = os.path.join(data_root, "data_object_image_2", split, "image_2")
    right_dir = os.path.join(data_root, "data_object_image_3", split, "image_3")

    calib_files = sorted(glob.glob(os.path.join(calib_dir, "*.txt")))
    print(f"found {len(calib_files)} calib files in {split}")

    lines = []
    skipped = 0
    for calib_path in calib_files:
        frame_id = os.path.splitext(os.path.basename(calib_path))[0]
        left_path = os.path.join(left_dir, frame_id + ".png")
        right_path = os.path.join(right_dir, frame_id + ".png")

        if not (os.path.exists(left_path) and os.path.exists(right_path)):
            skipped += 1
            continue
        try:
            focal, baseline = parse_calib(calib_path)
        except Exception:
            skipped += 1
            continue

        rel_left = os.path.relpath(left_path, data_root)
        rel_right = os.path.relpath(right_path, data_root)
        lines.append(f"{rel_left} {rel_right} {focal:.6f} {baseline:.6f}")

    print(f"skipped {skipped} incomplete samples")
    with open(out_path, "w") as f:
        f.write("\n".join(lines))
    print(f"wrote {len(lines)} samples to {out_path}")
    return lines


all_lines = build_list_file(CFG["data_root"],
                              os.path.join(CFG["work_dir"], "all_list.txt"),
                              split="training")

random.seed(CFG["seed"])
shuffled = all_lines.copy()
random.shuffle(shuffled)

n_val = int(len(shuffled) * CFG["val_ratio"])
n_test = int(len(shuffled) * CFG["test_ratio"])

val_lines = shuffled[:n_val]
test_lines = shuffled[n_val:n_val + n_test]
train_lines = shuffled[n_val + n_test:]

for name, lines in [("train", train_lines), ("val", val_lines), ("test", test_lines)]:
    with open(os.path.join(CFG["work_dir"], f"{name}_list.txt"), "w") as f:
        f.write("\n".join(lines))

print(f"train: {len(train_lines)}, val: {len(val_lines)}, test: {len(test_lines)}")
print("sample line:", train_lines[0])

In [2]:
# ============================================================
# CELL 3: HOG channel helper (cv2-based, เร็วกว่า skimage มาก)
# ============================================================
def compute_hog_channel(pil_img, img_h, img_w):
    """คืน per-cell HOG magnitude map resize เป็น full resolution, ค่า normalize 0-1
    ใช้ cv2.HOGDescriptor แทน skimage.feature.hog เพราะเร็วกว่ามาก (ไม่มี visualize overhead)"""
    gray = np.array(pil_img.convert("L").resize((img_w, img_h)))

    win_w = (img_w // 8) * 8
    win_h = (img_h // 8) * 8
    win_size = (win_w, win_h)
    gray_cropped = cv2.resize(gray, win_size)

    hog = cv2.HOGDescriptor(
        _winSize=win_size,
        _blockSize=(16, 16),
        _blockStride=(8, 8),
        _cellSize=(8, 8),
        _nbins=9,
    )
    feat = hog.compute(gray_cropped)

    n_cells_x = win_w // 8 - 1
    n_cells_y = win_h // 8 - 1
    feat_map = feat.reshape(n_cells_y, n_cells_x, -1).mean(axis=2)
    feat_map = cv2.resize(feat_map, (img_w, img_h))
    feat_map = feat_map / (feat_map.max() + 1e-8)
    return feat_map.astype(np.float32)

In [ ]:
# ============================================================
# CELL 4 (แก้ใหม่): SIFT precompute ด้วย multiprocessing (เร็วกว่าเดิมหลายเท่า)
# ============================================================
import multiprocessing as mp
from functools import partial

def build_sift_sparse_disp(left_gray, right_gray, img_w,
                             ratio_thresh=0.75, y_diff_thresh=2.0, max_disp_frac=0.5):
    sift = cv2.SIFT_create(nfeatures=500)   # จำกัดจำนวน feature -> เร็วขึ้น ไม่ต้องหา keypoint ทุกจุด
    kp_l, des_l = sift.detectAndCompute(left_gray, None)
    kp_r, des_r = sift.detectAndCompute(right_gray, None)

    H, W = left_gray.shape
    sparse_disp = np.zeros((H, W), dtype=np.float32)
    mask = np.zeros((H, W), dtype=bool)

    if des_l is None or des_r is None or len(kp_l) < 8 or len(kp_r) < 8:
        return sparse_disp, mask

    bf = cv2.BFMatcher()
    raw_matches = bf.knnMatch(des_l, des_r, k=2)

    good_matches = []
    for pair in raw_matches:
        if len(pair) != 2:
            continue
        m, n = pair
        if m.distance < ratio_thresh * n.distance:
            good_matches.append(m)

    for m in good_matches:
        xl, yl = kp_l[m.queryIdx].pt
        xr, yr = kp_r[m.trainIdx].pt
        if abs(yl - yr) > y_diff_thresh:
            continue
        disp_px = xl - xr
        if disp_px <= 0 or disp_px > W * max_disp_frac:
            continue
        yi, xi = int(round(yl)), int(round(xl))
        if 0 <= yi < H and 0 <= xi < W:
            sparse_disp[yi, xi] = disp_px / W
            mask[yi, xi] = True

    if mask.sum() < 8:
        return np.zeros((H, W), dtype=np.float32), np.zeros((H, W), dtype=bool)

    return sparse_disp, mask


def _process_one(args, data_root, cache_dir, img_h, img_w):
    i, line = args
    cache_path = os.path.join(cache_dir, f"{i:06d}.npz")
    if os.path.exists(cache_path):
        return i, "skipped"

    parts = line.split()
    if len(parts) < 2:
        return i, "invalid_line"

    left_rel, right_rel = parts[0], parts[1]
    left_img = cv2.imread(os.path.join(data_root, left_rel), cv2.IMREAD_GRAYSCALE)
    right_img = cv2.imread(os.path.join(data_root, right_rel), cv2.IMREAD_GRAYSCALE)

    if left_img is None or right_img is None:
        return i, "read_failed"

    left_img = cv2.resize(left_img, (img_w, img_h))
    right_img = cv2.resize(right_img, (img_w, img_h))

    sparse_disp, mask = build_sift_sparse_disp(left_img, right_img, img_w)
    np.savez_compressed(cache_path, disp=sparse_disp, mask=mask)
    return i, "done"


def precompute_sift_cache_parallel(data_root, list_lines, cache_dir, img_h, img_w, n_workers=None):
    os.makedirs(cache_dir, exist_ok=True)
    if n_workers is None:
        n_workers = max(mp.cpu_count() - 1, 1)   # เหลือ 1 core ให้ระบบอื่นทำงาน

    worker_fn = partial(_process_one, data_root=data_root, cache_dir=cache_dir,
                          img_h=img_h, img_w=img_w)
    tasks = list(enumerate(list_lines))

    print(f"เริ่ม precompute ด้วย {n_workers} process, {len(tasks)} ภาพ")

    with mp.Pool(processes=n_workers) as pool:
        for count, (i, status) in enumerate(pool.imap_unordered(worker_fn, tasks, chunksize=8)):
            if count % 200 == 0:
                print(f"[sift cache] {count}/{len(tasks)} ({status})")

    print(f"[sift cache] done: {cache_dir} ({len(tasks)} samples)")


# รันแบบ parallel แทนของเดิม
precompute_sift_cache_parallel(CFG["data_root"], train_lines,
                                 os.path.join(CFG["sift_cache_dir"], "train"),
                                 CFG["img_h"], CFG["img_w"])
precompute_sift_cache_parallel(CFG["data_root"], val_lines,
                                 os.path.join(CFG["sift_cache_dir"], "val"),
                                 CFG["img_h"], CFG["img_w"])
precompute_sift_cache_parallel(CFG["data_root"], test_lines,
                                 os.path.join(CFG["sift_cache_dir"], "test"),
                                 CFG["img_h"], CFG["img_w"])

In [3]:
# ============================================================
# CELL 5: DATASET (stereo pair + optional HOG channel + SIFT anchor cache)
# ============================================================
class StereoPairDataset(Dataset):
    def __init__(self, root, list_path, img_h, img_w, augment=True,
                 use_hog=False, sift_cache_dir=None):
        self.root = root
        self.img_h = img_h
        self.img_w = img_w
        self.augment = augment
        self.use_hog = use_hog
        self.sift_cache_dir = sift_cache_dir
        self.samples = []

        with open(list_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 4:
                    continue
                left_path, right_path, focal, baseline = parts[:4]
                self.samples.append({
                    "left": left_path,
                    "right": right_path,
                    "focal": float(focal),
                    "baseline": float(baseline),
                })

        self.resize = T.Resize((img_h, img_w))
        self.to_tensor = T.ToTensor()
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406],
                                       std=[0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.samples)

    def _load_img(self, path):
        img = Image.open(os.path.join(self.root, path)).convert("RGB")
        orig_w, orig_h = img.size
        img = self.resize(img)
        return img, orig_w, orig_h

    def _load_sift(self, idx):
        if self.sift_cache_dir is None:
            zeros_disp = np.zeros((self.img_h, self.img_w), dtype=np.float32)
            zeros_mask = np.zeros((self.img_h, self.img_w), dtype=bool)
            return zeros_disp, zeros_mask
        cache_path = os.path.join(self.sift_cache_dir, f"{idx:06d}.npz")
        if not os.path.exists(cache_path):
            zeros_disp = np.zeros((self.img_h, self.img_w), dtype=np.float32)
            zeros_mask = np.zeros((self.img_h, self.img_w), dtype=bool)
            return zeros_disp, zeros_mask
        data = np.load(cache_path)
        return data["disp"], data["mask"]

    def __getitem__(self, idx):
        s = self.samples[idx]
        left, orig_w, orig_h = self._load_img(s["left"])
        right, _, _ = self._load_img(s["right"])

        scale_w = self.img_w / orig_w
        focal = s["focal"] * scale_w
        baseline = s["baseline"]

        flipped = False
        if self.augment and random.random() < 0.5:
            left, right = right, left
            left = left.transpose(Image.FLIP_LEFT_RIGHT)
            right = right.transpose(Image.FLIP_LEFT_RIGHT)
            flipped = True

        if self.augment and random.random() < 0.5:
            jitter = T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
            seed = random.randint(0, 99999)
            torch.manual_seed(seed)
            left = jitter(left)
            torch.manual_seed(seed)
            right = jitter(right)

        left_t = self.normalize(self.to_tensor(left))
        right_t = self.normalize(self.to_tensor(right))

        if self.use_hog:
            hog_left = compute_hog_channel(left, self.img_h, self.img_w)
            hog_right = compute_hog_channel(right, self.img_h, self.img_w)
            left_t = torch.cat([left_t, torch.from_numpy(hog_left).unsqueeze(0)], dim=0)
            right_t = torch.cat([right_t, torch.from_numpy(hog_right).unsqueeze(0)], dim=0)

        sift_disp, sift_mask = self._load_sift(idx)
        # หมายเหตุ: ถ้า augment flip ภาพแล้ว sift cache (คำนวณจากภาพไม่ flip) จะไม่ตรงตำแหน่งกันเป๊ะ
        # เพื่อความปลอดภัย ปิด sift สำหรับ sample ที่ถูก flip (ให้ mask เป็น 0 ทั้งภาพ)
        if flipped:
            sift_disp = np.zeros_like(sift_disp)
            sift_mask = np.zeros_like(sift_mask)

        return {
            "left": left_t,
            "right": right_t,
            "focal": torch.tensor(focal, dtype=torch.float32),
            "baseline": torch.tensor(baseline, dtype=torch.float32),
            "sift_disp": torch.from_numpy(sift_disp).float(),
            "sift_mask": torch.from_numpy(sift_mask).bool(),
        }

In [4]:
# ============================================================
# CELL 6: MODEL (ConvNeXt V2 backbone + HOG late-fusion + bin head + GNN refiner)
# ============================================================
try:
    from torch_geometric.nn import GCNConv, knn_graph
    _HAS_PYG = True
except ImportError:
    _HAS_PYG = False
    print("⚠️ torch_geometric ไม่พบ -> ติดตั้งด้วย: pip install torch-geometric --break-system-packages")


class ConvNeXtV2Backbone(nn.Module):
    """เล็กกว่า ConvNeXt-T เดิม, pretrain ด้วย FCMAE (self-supervised) ให้ fine-tune quality ดีกว่า
    รันได้ทุก backend (CPU/GPU) ไม่มี dependency พิเศษ"""
    def __init__(self, name=CFG["backbone_name"], pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            name, pretrained=pretrained, features_only=True,
            out_indices=(0, 1, 2, 3)
        )
        self.out_channels = self.backbone.feature_info.channels()

    def forward(self, x):
        return self.backbone(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.act = nn.ELU(inplace=True)

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        if skip is not None:
            if skip.shape[-2:] != x.shape[-2:]:
                skip = F.interpolate(skip, size=x.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        x = self.act(self.bn1(self.conv1(x)))
        x = self.act(self.bn2(self.conv2(x)))
        return x


class HogEncoder(nn.Module):
    """Encoder เล็กแยกสำหรับ HOG channel เดียว (ไม่ใช่ full backbone -> compute ต่ำ)"""
    def __init__(self, base_ch=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, base_ch, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, base_ch * 2, 3, stride=2, padding=1), nn.ReLU(inplace=True),
        )
        self.out_ch = base_ch * 2

    def forward(self, hog):
        return self.conv(hog)


class HogProjector(nn.Module):
    """zero-init: ตอนเริ่มเทรน ไม่มีผลอะไรเลย (เหมือนไม่มี HOG) ป้องกันไม่ให้รบกวน pretrained decoder"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, 1)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, hog_feat, target_size):
        hog_resized = F.interpolate(hog_feat, size=target_size, mode="bilinear", align_corners=False)
        return self.proj(hog_resized)


class BinPredictor(nn.Module):
    def __init__(self, in_ch, n_bins, min_bin_width=1e-3):
        super().__init__()
        self.n_bins = n_bins
        self.min_bin_width = min_bin_width
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(in_ch, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, n_bins),
        )

    def forward(self, bottleneck_feat):
        B = bottleneck_feat.shape[0]
        x = self.pool(bottleneck_feat).view(B, -1)
        raw_widths = self.mlp(x)
        widths = F.softmax(raw_widths, dim=1)
        widths = self.min_bin_width + (1 - self.min_bin_width * self.n_bins) * widths
        edges = torch.cumsum(widths, dim=1)
        edges = F.pad(edges, (1, 0), value=0.0)
        centers = 0.5 * (edges[:, :-1] + edges[:, 1:])
        return centers


class BinDepthHead(nn.Module):
    def __init__(self, in_ch, n_bins):
        super().__init__()
        self.n_bins = n_bins
        self.conv = nn.Conv2d(in_ch, n_bins, 3, padding=1)

    def forward(self, x, bin_centers):
        logits = self.conv(x)
        probs = F.softmax(logits, dim=1)
        centers = bin_centers.view(bin_centers.shape[0], self.n_bins, 1, 1)
        disp = torch.sum(probs * centers, dim=1, keepdim=True)
        return disp, probs


class GraphDepthRefiner(nn.Module):
    """Graph propagation บน grid adjacency ตรงๆ (ไม่ใช้ knn_graph -> ไม่ต้องพึ่ง pyg-lib/torch-cluster)"""
    def __init__(self, feat_dim, hidden_dim=64, grid_stride=8, k_neighbors=8):
        super().__init__()
        self.grid_stride = grid_stride
        self.k_neighbors = k_neighbors  # เก็บไว้เผื่ออนาคต ไม่ได้ใช้แล้วตอนนี้
        self.gcn1 = GCNConv(feat_dim + 3, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, hidden_dim)
        self.gcn3 = GCNConv(hidden_dim, 1)
        self._cached_grid = {}   # cache edge_index ต่อ (H, W) เพราะ grid size คงที่ทุกภาพ ไม่ต้องสร้างใหม่ทุกครั้ง

    def build_node_graph(self, H, W, device):
        key = (H, W)
        if key in self._cached_grid:
            coords, edge_index = self._cached_grid[key]
            return coords.to(device), edge_index.to(device)

        n_rows = H // self.grid_stride
        n_cols = W // self.grid_stride

        ys, xs = torch.meshgrid(
            torch.arange(0, H, self.grid_stride),
            torch.arange(0, W, self.grid_stride),
            indexing="ij"
        )
        coords = torch.stack([xs.flatten(), ys.flatten()], dim=1).float()  # [N, 2]

        # สร้าง edge จาก grid adjacency 8-connected (แต่ละ node ต่อกับเพื่อนบ้านรอบทิศ)
        node_idx = torch.arange(n_rows * n_cols).view(n_rows, n_cols)
        edges = []
        for dy in [-1, 0, 1]:
            for dx in [-1, 0, 1]:
                if dy == 0 and dx == 0:
                    continue
                src = node_idx[max(0, -dy):n_rows - max(0, dy), max(0, -dx):n_cols - max(0, dx)]
                dst = node_idx[max(0, dy):n_rows - max(0, -dy), max(0, dx):n_cols - max(0, -dx)]
                edges.append(torch.stack([src.flatten(), dst.flatten()], dim=0))

        edge_index = torch.cat(edges, dim=1)  # [2, num_edges]

        self._cached_grid[key] = (coords, edge_index)
        return coords.to(device), edge_index.to(device)

    def forward(self, decoder_feat, coarse_disp, sift_disp, sift_mask):
        B, C, H, W = decoder_feat.shape
        device = decoder_feat.device
        coords, edge_index = self.build_node_graph(H, W, device)
        gy = coords[:, 1].long()
        gx = coords[:, 0].long()

        refined_disps = []
        for b in range(B):
            node_feat = decoder_feat[b, :, gy, gx].T
            node_coarse_disp = coarse_disp[b, 0, gy, gx].unsqueeze(1)
            node_sift_disp = sift_disp[b, gy, gx].unsqueeze(1)
            node_sift_mask = sift_mask[b, gy, gx].float().unsqueeze(1)

            x = torch.cat([node_feat, node_coarse_disp, node_sift_disp, node_sift_mask], dim=1)
            x = F.relu(self.gcn1(x, edge_index))
            x = F.relu(self.gcn2(x, edge_index))
            residual = self.gcn3(x, edge_index)

            residual_grid = residual.view(1, 1, H // self.grid_stride, W // self.grid_stride)
            residual_full = F.interpolate(residual_grid, size=(H, W), mode="bilinear", align_corners=False)
            refined_disps.append(residual_full)

        residual_full_batch = torch.cat(refined_disps, dim=0)
        refined_disp = torch.clamp(coarse_disp + residual_full_batch, 0.0, 1.0)
        return refined_disp


class MonolithicDApe(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = ConvNeXtV2Backbone()
        c = self.backbone.out_channels
        n_bins = CFG["n_bins"]

        self.dec4 = DecoderBlock(c[3], c[2], 256)
        self.dec3 = DecoderBlock(256, c[1], 128)
        self.dec2 = DecoderBlock(128, c[0], 64)
        self.dec1 = DecoderBlock(64, 0, 32)

        self.bin_predictor = BinPredictor(c[3], n_bins)
        self.head4 = BinDepthHead(256, n_bins)
        self.head3 = BinDepthHead(128, n_bins)
        self.head2 = BinDepthHead(64, n_bins)
        self.head1 = BinDepthHead(32, n_bins)

        self.use_hog = CFG["use_hog"]
        if self.use_hog:
            self.hog_encoder = HogEncoder(base_ch=32)
            hog_ch = self.hog_encoder.out_ch
            self.hog_proj_dec2 = HogProjector(hog_ch, 64)
            self.hog_proj_dec1 = HogProjector(hog_ch, 32)

        self.use_gnn_refine = CFG["use_gnn_refine"] and _HAS_PYG
        if CFG["use_gnn_refine"] and not _HAS_PYG:
            print("⚠️ use_gnn_refine=True แต่ torch_geometric ไม่พร้อมใช้งาน -> ปิด GNN refinement ชั่วคราว")
        if self.use_gnn_refine:
            self.graph_refiner = GraphDepthRefiner(feat_dim=32, hidden_dim=64,
                                                      grid_stride=CFG["gnn_grid_stride"],
                                                      k_neighbors=CFG["gnn_k_neighbors"])

    def freeze_backbone(self, freeze=True):
        for p in self.backbone.parameters():
            p.requires_grad = not freeze

    def forward(self, x, return_probs=False, sift_disp=None, sift_mask=None):
        rgb = x[:, :3]
        f1, f2, f3, f4 = self.backbone(rgb)
        bin_centers = self.bin_predictor(f4)

        hog_feat = None
        if self.use_hog:
            hog = x[:, 3:4]
            hog_feat = self.hog_encoder(hog)

        d4 = self.dec4(f4, f3)
        disp4, probs4 = self.head4(d4, bin_centers)

        d3 = self.dec3(d4, f2)
        disp3, probs3 = self.head3(d3, bin_centers)

        d2 = self.dec2(d3, f1)
        if hog_feat is not None:
            d2 = d2 + self.hog_proj_dec2(hog_feat, d2.shape[-2:])
        disp2, probs2 = self.head2(d2, bin_centers)

        d1 = self.dec1(d2, None)
        if hog_feat is not None:
            d1 = d1 + self.hog_proj_dec1(hog_feat, d1.shape[-2:])
        disp1, probs1 = self.head1(d1, bin_centers)

        if self.use_gnn_refine and sift_disp is not None and sift_mask is not None:
            disp1 = self.graph_refiner(d1, disp1, sift_disp, sift_mask)

        disps = [disp1, disp2, disp3, disp4]
        if return_probs:
            return disps, [probs1, probs2, probs3, probs4], bin_centers
        return disps

⚠️ torch_geometric ไม่พบ -> ติดตั้งด้วย: pip install torch-geometric --break-system-packages


In [5]:
# ============================================================
# CELL 7: LOSS FUNCTIONS (photometric auto-mask + smoothness + LR consistency + bin entropy)
# ============================================================
def disp_to_depth(disp, min_depth, max_depth):
    min_disp = 1.0 / max_depth
    max_disp = 1.0 / min_depth
    scaled_disp = min_disp + (max_disp - min_disp) * disp
    depth = 1.0 / scaled_disp
    return depth


def warp_image(img, disp):
    B, C, H, W = img.shape
    pixel_disp = disp * W

    xs = torch.linspace(0, W - 1, W, device=img.device).view(1, 1, 1, W).expand(B, 1, H, W)
    ys = torch.linspace(0, H - 1, H, device=img.device).view(1, 1, H, 1).expand(B, 1, H, W)

    x_shifted = xs - pixel_disp
    x_norm = 2.0 * x_shifted / (W - 1) - 1.0
    y_norm = 2.0 * ys / (H - 1) - 1.0

    grid = torch.cat([x_norm.permute(0, 2, 3, 1), y_norm.permute(0, 2, 3, 1)], dim=-1)
    warped = F.grid_sample(img, grid, mode="bilinear", padding_mode="border", align_corners=False)
    return warped


def ssim(x, y):
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    pad = nn.ReflectionPad2d(1)
    x = pad(x)
    y = pad(y)
    mu_x = F.avg_pool2d(x, 3, 1)
    mu_y = F.avg_pool2d(y, 3, 1)
    sigma_x = F.avg_pool2d(x ** 2, 3, 1) - mu_x ** 2
    sigma_y = F.avg_pool2d(y ** 2, 3, 1) - mu_y ** 2
    sigma_xy = F.avg_pool2d(x * y, 3, 1) - mu_x * mu_y
    ssim_n = (2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)
    ssim_d = (mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x + sigma_y + C2)
    ssim_map = ssim_n / ssim_d
    return torch.clamp((1 - ssim_map) / 2, 0, 1)


def photometric_loss(pred, target, ssim_weight=CFG["ssim_weight"]):
    l1 = torch.abs(pred - target).mean(1, keepdim=True)
    ssim_loss = ssim(pred, target).mean(1, keepdim=True)
    return ssim_weight * ssim_loss + (1 - ssim_weight) * l1


def smoothness_loss(disp, img):
    grad_disp_x = torch.abs(disp[:, :, :, :-1] - disp[:, :, :, 1:])
    grad_disp_y = torch.abs(disp[:, :, :-1, :] - disp[:, :, 1:, :])

    grad_img_x = torch.mean(torch.abs(img[:, :, :, :-1] - img[:, :, :, 1:]), 1, keepdim=True)
    grad_img_y = torch.mean(torch.abs(img[:, :, :-1, :] - img[:, :, 1:, :]), 1, keepdim=True)

    grad_disp_x = grad_disp_x * torch.exp(-grad_img_x)
    grad_disp_y = grad_disp_y * torch.exp(-grad_img_y)

    return grad_disp_x.mean() + grad_disp_y.mean()


def bin_usage_loss(probs):
    """ต่างจาก bin_entropy_loss: ตัวนี้มองภาพรวมทั้ง batch ไม่ใช่ต่อ pixel
    บังคับให้ 'ค่าเฉลี่ยการใช้งาน' ของแต่ละ bin กระจายมากขึ้น (สูง entropy ของ marginal distribution)
    ป้องกัน bin ส่วนใหญ่ไม่ถูกใช้เลยทั้งภาพ/ทั้ง batch"""
    mean_probs = probs.mean(dim=(0, 2, 3))  # [n_bins] เฉลี่ยข้าม batch+spatial
    usage_entropy = -(mean_probs * torch.log(mean_probs + 1e-8)).sum()
    return -usage_entropy  # ต้องการ entropy สูง -> loss คือค่าลบของมัน (minimize loss = maximize entropy)


def compute_loss(disps_left, disps_right, left_img, right_img, probs_left=None, probs_right=None):
    total_loss = 0.0
    num_scales = len(disps_left)

    for scale_idx, (disp_l, disp_r) in enumerate(zip(disps_left, disps_right)):
        h, w = disp_l.shape[-2:]
        img_l = F.interpolate(left_img, size=(h, w), mode="bilinear", align_corners=False)
        img_r = F.interpolate(right_img, size=(h, w), mode="bilinear", align_corners=False)

        warped_l = warp_image(img_r, disp_l)
        warped_r = warp_image(img_l, -disp_r)

        photo_l = photometric_loss(warped_l, img_l)
        photo_r = photometric_loss(warped_r, img_r)

        identity_loss_l = photometric_loss(img_r, img_l)
        identity_loss_r = photometric_loss(img_l, img_r)

        mask_l = (photo_l < identity_loss_l).float()
        mask_r = (photo_r < identity_loss_r).float()

        photo_l_masked = (photo_l * mask_l).sum() / (mask_l.sum() + 1e-8)
        photo_r_masked = (photo_r * mask_r).sum() / (mask_r.sum() + 1e-8)

        smooth_l = smoothness_loss(disp_l, img_l)
        smooth_r = smoothness_loss(disp_r, img_r)

        warped_disp_r = warp_image(disp_r, disp_l)
        lr_consistency = torch.abs(disp_l - warped_disp_r).mean()

        scale_loss = (
            photo_l_masked + photo_r_masked
            + CFG["smoothness_weight"] * (smooth_l + smooth_r) / (2 ** scale_idx)
            + CFG["lr_consistency_weight"] * lr_consistency
        )
        total_loss = total_loss + scale_loss / num_scales

    if probs_left is not None and probs_right is not None:
        entropy_l = bin_entropy_loss(probs_left[0])
        entropy_r = bin_entropy_loss(probs_right[0])
        total_loss = total_loss + CFG["bin_entropy_weight"] * (entropy_l + entropy_r) / 2

        # เพิ่มใหม่: usage diversity loss
        usage_l = bin_usage_loss(probs_left[0])
        usage_r = bin_usage_loss(probs_right[0])
        total_loss = total_loss + CFG["bin_usage_weight"] * (usage_l + usage_r) / 2

    return total_loss

In [6]:
# ============================================================
# CELL 8: OPTIMIZER (AdamW + cosine schedule, แยก weight decay) + EVALUATE
# ============================================================
def build_optimizer(model, freeze_backbone):
    model.freeze_backbone(freeze_backbone)

    if freeze_backbone:
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=CFG["lr_head"], weight_decay=1e-4)
    else:
        backbone_decay, backbone_no_decay = [], []
        for n, p in model.backbone.named_parameters():
            (backbone_no_decay if ("norm" in n or "bias" in n) else backbone_decay).append(p)

        head_decay, head_no_decay = [], []
        for n, p in model.named_parameters():
            if n.startswith("backbone"):
                continue
            (head_no_decay if ("norm" in n or "bias" in n) else head_decay).append(p)

        optimizer = torch.optim.AdamW([
            {"params": backbone_decay, "lr": CFG["lr_backbone"], "weight_decay": 1e-4},
            {"params": backbone_no_decay, "lr": CFG["lr_backbone"], "weight_decay": 0.0},
            {"params": head_decay, "lr": CFG["lr_head"], "weight_decay": 1e-4},
            {"params": head_no_decay, "lr": CFG["lr_head"], "weight_decay": 0.0},
        ])

    remaining_epochs = max(CFG["epochs"] - CFG["freeze_backbone_epochs"], 1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=remaining_epochs, eta_min=1e-6
    )
    return optimizer, scheduler


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    n = 0
    for batch in loader:
        left = batch["left"].to(device, non_blocking=True)
        right = batch["right"].to(device, non_blocking=True)
        sift_disp = batch["sift_disp"].to(device, non_blocking=True)
        sift_mask = batch["sift_mask"].to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=CFG["amp"]):
            disps_left, probs_left, _ = model(left, return_probs=True,
                                                sift_disp=sift_disp, sift_mask=sift_mask)
            disps_right, probs_right, _ = model(right, return_probs=True,
                                                  sift_disp=sift_disp, sift_mask=sift_mask)
            loss = compute_loss(disps_left, disps_right, left[:, :3], right[:, :3],
                                 probs_left, probs_right)
        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)

In [ ]:
# ============================================================
# CELL 9: MOCKUP TEST 1 — Dataset sanity check
# ============================================================
mock_ds = StereoPairDataset(
    CFG["data_root"],
    os.path.join(CFG["work_dir"], "train_list.txt"),
    CFG["img_h"], CFG["img_w"], augment=True,
    use_hog=CFG["use_hog"],
    sift_cache_dir=os.path.join(CFG["sift_cache_dir"], "train"),
)

print("dataset length:", len(mock_ds))
sample = mock_ds[0]
print("left shape:", sample["left"].shape)
print("right shape:", sample["right"].shape)
print("focal:", sample["focal"].item(), "baseline:", sample["baseline"].item())
print("sift_disp valid pixels:", sample["sift_mask"].sum().item())

expected_channels = 4 if CFG["use_hog"] else 3
assert sample["left"].shape[0] == expected_channels, \
    f"channel mismatch: ได้ {sample['left'].shape[0]} channel, คาดว่า {expected_channels}"
assert not torch.isnan(sample["left"]).any(), "NaN in left image"
assert not torch.isnan(sample["right"]).any(), "NaN in right image"
assert sample["focal"].item() > 0, "focal ต้องเป็นค่าบวก"
assert 0.3 < sample["baseline"].item() < 0.8, f"baseline ผิดปกติ: {sample['baseline'].item()}"
print("dataset sanity check: PASS")

In [ ]:
# ============================================================
# CELL 10: MOCKUP TEST 2 — Model forward shape + VRAM check
# ============================================================
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = MonolithicDApe().to(CFG["device"])
model.eval()

in_channels = 4 if CFG["use_hog"] else 3
dummy_batch = torch.randn(CFG["batch_size"], in_channels, CFG["img_h"], CFG["img_w"]).to(CFG["device"])
dummy_sift_disp = torch.zeros(CFG["batch_size"], CFG["img_h"], CFG["img_w"]).to(CFG["device"])
dummy_sift_mask = torch.zeros(CFG["batch_size"], CFG["img_h"], CFG["img_w"], dtype=torch.bool).to(CFG["device"])

with torch.no_grad():
    disps = model(dummy_batch, sift_disp=dummy_sift_disp, sift_mask=dummy_sift_mask)

print("จำนวน output scale:", len(disps))
for i, d in enumerate(disps):
    print(f"  scale {i} shape: {d.shape}, range: [{d.min().item():.4f}, {d.max().item():.4f}]")

peak_mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"peak VRAM (forward only, batch={CFG['batch_size']}): {peak_mem:.2f} GB")
print("model forward shape check: PASS")

In [ ]:
# ============================================================
# CELL 11: MOCKUP TEST 3 — Loss computation + backward check
# ============================================================
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model.train()
mock_loader = DataLoader(mock_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=0)
batch = next(iter(mock_loader))

left = batch["left"].to(CFG["device"])
right = batch["right"].to(CFG["device"])
sift_disp = batch["sift_disp"].to(CFG["device"])
sift_mask = batch["sift_mask"].to(CFG["device"])

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr_head"])
scaler = torch.amp.GradScaler("cuda", enabled=CFG["amp"])

optimizer.zero_grad()
with torch.amp.autocast("cuda", enabled=CFG["amp"]):
    disps_left, probs_left, _ = model(left, return_probs=True, sift_disp=sift_disp, sift_mask=sift_mask)
    disps_right, probs_right, _ = model(right, return_probs=True, sift_disp=sift_disp, sift_mask=sift_mask)
    loss = compute_loss(disps_left, disps_right, left[:, :3], right[:, :3], probs_left, probs_right)

print("loss value:", loss.item())
assert not torch.isnan(loss), "loss เป็น NaN!"
assert not torch.isinf(loss), "loss เป็น Inf!"

scaler.scale(loss).backward()

has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.backbone.parameters())
print("gradient ไหลถึง backbone:", has_grad)

scaler.step(optimizer)
scaler.update()

peak_mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"peak VRAM (forward+backward, batch={CFG['batch_size']}): {peak_mem:.2f} GB")
print("loss + backward check: PASS")

In [ ]:
# ============================================================
# CELL 12: MOCKUP TEST 4 — Mini overfit sanity check
# ============================================================
import time

tiny_indices = list(range(min(16, len(mock_ds))))
tiny_subset = torch.utils.data.Subset(mock_ds, tiny_indices)
tiny_loader = DataLoader(tiny_subset, batch_size=min(4, CFG["batch_size"]), shuffle=True, num_workers=0)

mini_model = MonolithicDApe().to(CFG["device"])
mini_model.train()
mini_opt = torch.optim.AdamW(mini_model.parameters(), lr=1e-4)
mini_scaler = torch.amp.GradScaler("cuda", enabled=CFG["amp"])

loss_history = []
t0 = time.time()
n_steps = 60

for step in range(n_steps):
    batch = next(iter(tiny_loader))
    left = batch["left"].to(CFG["device"])
    right = batch["right"].to(CFG["device"])
    sift_disp = batch["sift_disp"].to(CFG["device"])
    sift_mask = batch["sift_mask"].to(CFG["device"])

    mini_opt.zero_grad()
    with torch.amp.autocast("cuda", enabled=CFG["amp"]):
        dl, pl, _ = mini_model(left, return_probs=True, sift_disp=sift_disp, sift_mask=sift_mask)
        dr, pr, _ = mini_model(right, return_probs=True, sift_disp=sift_disp, sift_mask=sift_mask)
        loss = compute_loss(dl, dr, left[:, :3], right[:, :3], pl, pr)

    mini_scaler.scale(loss).backward()
    mini_scaler.step(mini_opt)
    mini_scaler.update()

    loss_history.append(loss.item())
    if step % 10 == 0:
        print(f"step {step}: loss = {loss.item():.4f}")

elapsed = time.time() - t0
print(f"\nเวลารวม {n_steps} step: {elapsed:.1f}s -> ~{elapsed/n_steps:.2f}s/step")
print(f"loss เริ่มต้น: {loss_history[0]:.4f}, loss ท้ายสุด: {loss_history[-1]:.4f}")

if loss_history[-1] < loss_history[0]:
    print("mini overfit test: PASS (loss ลดลง)")
else:
    print("⚠️ mini overfit test: loss ไม่ลดลง — เช็ค pipeline ก่อนรันเทรนจริง")

In [ ]:
# ============================================================
# CELL 13: FULL TRAINING LOOP (save ทุก save_every epoch + best only)
# ============================================================
def train():
    device = CFG["device"]
    expected_channels = 4 if CFG["use_hog"] else 3

    train_ds = StereoPairDataset(CFG["data_root"],
                                   os.path.join(CFG["work_dir"], "train_list.txt"),
                                   CFG["img_h"], CFG["img_w"], augment=True,
                                   use_hog=CFG["use_hog"],
                                   sift_cache_dir=os.path.join(CFG["sift_cache_dir"], "train"))
    val_ds = StereoPairDataset(CFG["data_root"],
                                 os.path.join(CFG["work_dir"], "val_list.txt"),
                                 CFG["img_h"], CFG["img_w"], augment=False,
                                 use_hog=CFG["use_hog"],
                                 sift_cache_dir=os.path.join(CFG["sift_cache_dir"], "val"))

    sample_check = train_ds[0]
    assert sample_check["left"].shape[0] == expected_channels, \
        f"channel mismatch: dataset ให้ {sample_check['left'].shape[0]} channel, คาดว่า {expected_channels}"
    print(f"channel check ผ่าน: {expected_channels} channel")

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                                num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                              num_workers=CFG["num_workers"], pin_memory=True)

    model = MonolithicDApe().to(device)
    scaler = torch.amp.GradScaler("cuda", enabled=CFG["amp"])

    optimizer = None
    scheduler = None
    current_phase_frozen = None
    best_val_loss = float("inf")

    for epoch in range(CFG["epochs"]):
        freeze = epoch < CFG["freeze_backbone_epochs"]
        if freeze != current_phase_frozen:
            torch.cuda.empty_cache()
            optimizer, scheduler = build_optimizer(model, freeze)
            current_phase_frozen = freeze
            print(f"[epoch {epoch}] backbone frozen = {freeze}, optimizer rebuilt")

        model.train()
        running_loss = 0.0

        for step, batch in enumerate(train_loader):
            left = batch["left"].to(device, non_blocking=True)
            right = batch["right"].to(device, non_blocking=True)
            sift_disp = batch["sift_disp"].to(device, non_blocking=True)
            sift_mask = batch["sift_mask"].to(device, non_blocking=True)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=CFG["amp"]):
                disps_left, probs_left, _ = model(left, return_probs=True,
                                                    sift_disp=sift_disp, sift_mask=sift_mask)
                disps_right, probs_right, _ = model(right, return_probs=True,
                                                      sift_disp=sift_disp, sift_mask=sift_mask)
                loss = compute_loss(disps_left, disps_right, left[:, :3], right[:, :3],
                                     probs_left, probs_right)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            if step % CFG["log_every"] == 0:
                print(f"epoch {epoch} step {step}/{len(train_loader)} loss {running_loss/(step+1):.4f}")

        train_loss = running_loss / len(train_loader)
        val_loss = evaluate(model, val_loader, device)
        print(f"=== epoch {epoch} done | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} ===")

        if scheduler is not None and not freeze:
            scheduler.step()
            print(f"  lr after step: {optimizer.param_groups[0]['lr']:.2e}")

        if (epoch + 1) % CFG["save_every"] == 0:
            ckpt_path = os.path.join(CFG["ckpt_dir"], f"depape_epoch{epoch}.pt")
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "train_loss": train_loss, "val_loss": val_loss}, ckpt_path)
            print(f"saved periodic checkpoint: {ckpt_path}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(CFG["ckpt_dir"], "depape_best.pt")
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "train_loss": train_loss, "val_loss": val_loss}, best_path)
            print(f"new best (val_loss={val_loss:.4f}), saved: {best_path}")

    return model

model = train()

In [ ]:
# ============================================================
# CELL 14: Inference helper (shared input-tensor builder + model loader)
# ============================================================
_infer_resize = T.Resize((CFG["img_h"], CFG["img_w"]))
_infer_to_tensor = T.ToTensor()
_infer_normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


def build_input_tensor(pil_img, img_h, img_w):
    """สร้าง input tensor sync กับ CFG['use_hog'] เสมอ ใช้ร่วมกันทุก inference cell"""
    resized = _infer_resize(pil_img) if pil_img.size != (img_w, img_h) else pil_img
    resized = pil_img.resize((img_w, img_h))
    rgb_t = _infer_normalize(_infer_to_tensor(resized))

    if CFG["use_hog"]:
        hog_map = compute_hog_channel(resized, img_h, img_w)
        hog_t = torch.from_numpy(hog_map).unsqueeze(0)
        return torch.cat([rgb_t, hog_t], dim=0)

    return rgb_t


CFG["custom_ckpt_path"] = None

ckpt_path = CFG["custom_ckpt_path"] if CFG["custom_ckpt_path"] and os.path.exists(CFG["custom_ckpt_path"]) \
            else os.path.join(CFG["ckpt_dir"], "depape_best.pt")

if CFG["custom_ckpt_path"] and not os.path.exists(CFG["custom_ckpt_path"]):
    print(f"⚠️ ไม่พบไฟล์ {CFG['custom_ckpt_path']} -> ใช้ depape_best.pt แทน")

ckpt = torch.load(ckpt_path, map_location=CFG["device"])
our_model = MonolithicDApe().to(CFG["device"])
our_model.load_state_dict(ckpt["model_state"])
our_model.eval()
print(f"loaded checkpoint: {ckpt_path}")
print(f"  epoch {ckpt['epoch']}, val_loss {ckpt['val_loss']:.4f}")


@torch.no_grad()
def infer_batch(model, pil_images, n_bins=CFG["n_bins"]):
    """รับ list ของ PIL image คืน (soft_disp, topo_map) ต่อภาพ ที่ full resolution
    หมายเหตุ: ไม่มี SIFT anchor ตอน inference บนภาพทั่วไป (ไม่ใช่ KITTI stereo)
    -> ส่ง sift_mask เป็นศูนย์ทั้งภาพ, GNN refiner จะทำงานแบบใช้แค่ visual prior"""
    tensors = torch.stack([
        build_input_tensor(img, CFG["img_h"], CFG["img_w"]) for img in pil_images
    ]).to(CFG["device"])

    B = tensors.shape[0]
    dummy_sift_disp = torch.zeros(B, CFG["img_h"], CFG["img_w"]).to(CFG["device"])
    dummy_sift_mask = torch.zeros(B, CFG["img_h"], CFG["img_w"], dtype=torch.bool).to(CFG["device"])

    disps, probs_list, bin_centers = model(tensors, return_probs=True,
                                             sift_disp=dummy_sift_disp, sift_mask=dummy_sift_mask)

    disp_full = F.interpolate(disps[0], size=(CFG["img_h"], CFG["img_w"]),
                               mode="bilinear", align_corners=False)
    probs_full = F.interpolate(probs_list[0], size=(CFG["img_h"], CFG["img_w"]),
                                mode="bilinear", align_corners=False)

    soft_disp_batch = disp_full.squeeze(1).cpu().numpy()
    bin_idx = probs_full.argmax(dim=1)
    topo_batch = (bin_idx.float() / (n_bins - 1)).cpu().numpy()

    return soft_disp_batch, topo_batch

In [ ]:
# ============================================================
# CELL 15: Topographic-style visualization (สุ่ม 5 รูป)
# ============================================================
import matplotlib.pyplot as plt
import random as _random

n_samples = 5
random_lines = _random.sample(train_lines, n_samples)

fig, axes = plt.subplots(n_samples, 3, figsize=(20, 5 * n_samples))

for row, line in enumerate(random_lines):
    left_rel = line.split()[0]
    raw_pil = Image.open(os.path.join(CFG["data_root"], left_rel)).convert("RGB")
    raw_img_np = np.array(raw_pil.resize((CFG["img_w"], CFG["img_h"])))

    soft_disp_batch, topo_batch = infer_batch(our_model, [raw_pil])
    soft_disp, topo_map = soft_disp_batch[0], topo_batch[0]

    axes[row, 0].imshow(raw_img_np, origin="upper", aspect="equal")
    axes[row, 0].set_title("input (left)" if row == 0 else "")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(soft_disp, cmap="magma", origin="upper", aspect="equal")
    axes[row, 1].set_title("soft-combine (continuous)" if row == 0 else "")
    axes[row, 1].axis("off")

    axes[row, 2].contourf(topo_map, levels=CFG["n_bins"] // 4, cmap="magma", origin="upper")
    axes[row, 2].set_aspect("equal")
    axes[row, 2].set_title(f"topographic ({CFG['n_bins']} bins)" if row == 0 else "")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

In [9]:
!kaggle kernels output itsmetinnaphop/gandape-network/ckpts -p /kaggle/working/

all_list.txt: Skipping, found more recently modified local copy (use --force to force download)
000000.npz: Skipping, found more recently modified local copy (use --force to force download)
000001.npz: Skipping, found more recently modified local copy (use --force to force download)
000002.npz: Skipping, found more recently modified local copy (use --force to force download)
000003.npz: Skipping, found more recently modified local copy (use --force to force download)
000004.npz: Skipping, found more recently modified local copy (use --force to force download)
000005.npz: Skipping, found more recently modified local copy (use --force to force download)
000006.npz: Skipping, found more recently modified local copy (use --force to force download)
000007.npz: Skipping, found more recently modified local copy (use --force to force download)
000008.npz: Skipping, found more recently modified local copy (use --force to force download)
000009.npz: Skipping, found more recently modified local c

In [12]:
# ============================================================
# CELL 18: Export เป็น ONNX สำหรับ client-side inference (ตัด HOG/GNN)
# ============================================================
import torch.onnx

web_model = MonolithicDApe()
web_model.use_hog = False
web_model.use_gnn_refine = False

ckpt = torch.load("/kaggle/input/models/itsmetinnaphop/v2ws/pytorch/default/1/depape_best (2).pt", map_location="cpu")
state = ckpt["model_state"]

# กรอง key ของ hog_encoder/hog_proj_*/graph_refiner ออก เพราะ web_model ไม่มี module พวกนี้แล้ว
filtered_state = {k: v for k, v in state.items()
                    if not k.startswith("hog_encoder")
                    and not k.startswith("hog_proj_")
                    and not k.startswith("graph_refiner")}

missing, unexpected = web_model.load_state_dict(filtered_state, strict=False)
print("missing keys (คาดว่าจะไม่มี ถ้า backbone/decoder/bin head ตรงกันหมด):", missing)
print("unexpected keys ที่ถูกข้าม:", unexpected)

web_model.eval()

dummy_input = torch.randn(1, 3, CFG["img_h"], CFG["img_w"])

torch.onnx.export(
    web_model, dummy_input,
    os.path.join(CFG["work_dir"], "monolithic_dape_web.onnx"),
    input_names=["input"],
    output_names=["disp1", "disp2", "disp3", "disp4"],
    dynamic_axes={"input": {0: "batch"}},
    opset_version=17,
)
print("exported: monolithic_dape_web.onnx")

⚠️ use_gnn_refine=True แต่ torch_geometric ไม่พร้อมใช้งาน -> ปิด GNN refinement ชั่วคราว
missing keys (คาดว่าจะไม่มี ถ้า backbone/decoder/bin head ตรงกันหมด): ['hog_encoder.conv.0.weight', 'hog_encoder.conv.0.bias', 'hog_encoder.conv.2.weight', 'hog_encoder.conv.2.bias', 'hog_proj_dec2.proj.weight', 'hog_proj_dec2.proj.bias', 'hog_proj_dec1.proj.weight', 'hog_proj_dec1.proj.bias']
unexpected keys ที่ถูกข้าม: []


/tmp/ipykernel_58/1948639676.py:27: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0708 12:18:22.373000 58 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0708 12:18:23.281000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, a

[torch.onnx] Obtain model graph for `MonolithicDApe([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MonolithicDApe([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py"

[torch.onnx] Translate the graph into ONNX... ✅


Skipping constant folding for op SequenceEmpty with multiple outputs.
Skipping constant folding for op SequenceEmpty with multiple outputs.


exported: monolithic_dape_web.onnx


In [11]:
!pip install onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.3 MB/s eta 0:00:00
